In [2]:
import os
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from pathlib import Path


In [3]:
base_path = "/home/jupyter/project/"
output_path = os.path.join(base_path, "blends")
leakage_path = os.path.join(base_path, "EDA/leakage_from_train.csv")

In [7]:
base_path = "/home/jupyter/project/"
output_path = os.path.join(base_path, "blends")
leakage_path = os.path.join(base_path, "EDA/leakage_from_train.csv")

files = {
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "lgb_eng": "submissions_v2/submission_lgb_top500_engineered.csv",  # будет добавлена утечка
    "tabm": "submissions/tabm_top500_magic_meta_0.65617.csv",          # будет добавлена утечка
    "cat": "submissions/old_catboost.csv",                             # будет добавлена утечка
    "flaml": "submissions/flaml_all_train_fields_0.66582.csv",         # будет добавлена утечка
}

index_col = "index"
target_col = "score"

# Загрузка данных утечки
leak_df = pd.read_csv(leakage_path)
# Превращаем в словарь {test_index: true_target} для быстрой замены
leak_dict = dict(zip(leak_df["test_index"], leak_df["true_target"]))

# Загрузка основных сабмитов
dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items()}

# Внедрение утечки
files_with_leak = ["lgb_eng", "tabm", "cat", "flaml"]
for name in files_with_leak:
    # Заменяем значения в target_col, если index есть в утечке
    dfs[name][target_col] = dfs[name][index_col].map(leak_dict).fillna(dfs[name][target_col])

# Извлечение рангов
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
base_df = dfs["lgb_eng"][[index_col]].copy()

# Блендинг рангов
name, final_rank = (
    "blend_01_lgbm_heavy_p4.5",
    0.2525 * (ranks["lgb_eng"] ** 4.5 + ranks["lgb_dart"] ** 4.5) + 
    0.165 * (ranks["tabm"] ** 4.5 + ranks["cat"] ** 4.5 + ranks["flaml"] ** 4.5),
)

# 6. Финальное ранжирование бленда
final_score = rankdata(final_rank) / len(final_rank)
sub = base_df.copy()
sub[target_col] = final_score
sub.to_csv(os.path.join(output_path, f"{name}.csv"), index=False)

print(f"Готово! Файл успешно сохранен в папку: {output_path}")

Готово! Файл успешно сохранен в папку: /home/jupyter/project/blends


In [5]:
files = {
    "lgb_eng": "submissions_v2/submission_lgb_top500_engineered.csv",
    "log_stacker_v3": "submissions_v3/submission_blend_logistic_stacker_v3.csv",
    "lgb_dart_v3": "submissions_v3/submission_lgb_dart_v3.csv",
    "lgb_dart_v2": "submissions_v2/submission_lgb_peak_dart.csv",
    "lgb_deep_v3": "submissions_v3/submission_lgb_deeper_v3.csv",
    "flaml_v3": "submissions_v3/submission_flaml_automl_v3.csv"
}

index_col = "index"
target_col = "score"

# 1. Загрузка чистых сабмитов
dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items()}

# 2. Извлечение рангов чистых моделей
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
base_df = dfs["lgb_eng"][[index_col]].copy()

# 3. Продвинутый блендинг рангов со степенным масштабированием
# Мы даем наибольший вес двум лидерам (инженерному LGBM и логистическому стекеру)
# И используем степень 4.0 - 5.0 для выделения уверенных предсказаний
name = "super_blend_leak_at_the_end_v1"

final_rank = (
    0.30 * (ranks["lgb_eng"] ** 4.5) + 
    0.25 * (ranks["log_stacker_v3"] ** 4.5) + 
    0.15 * (ranks["lgb_dart_v3"] ** 4.5) +
    0.12 * (ranks["lgb_dart_v2"] ** 4.5) +
    0.10 * (ranks["lgb_deep_v3"] ** 4.5) +
    0.08 * (ranks["flaml_v3"] ** 4.5)
)

# 4. Получение финального скора из бленда рангов
final_score = rankdata(final_rank) / len(final_rank)

sub = base_df.copy()
sub[target_col] = final_score

# Загружаем утечку
leak_df = pd.read_csv(leakage_path)
# Для рангов: истинный 1 должен получить максимальный ранг (1.0), а 0 - минимальный (0.0)
# Чтобы не портить непрерывное распределение, заменяем на крайние значения рангов
leak_dict = dict(zip(leak_df["test_index"], leak_df["true_target"]))

# Применяем утечку к итоговому результату
sub[target_col] = sub[index_col].map(leak_dict).fillna(sub[target_col])

# Сохраняем результат
os.makedirs(output_path, exist_ok=True)
sub.to_csv(os.path.join(output_path, f"{name}.csv"), index=False)

print(f"Супер-бленд готов! Файл сохранен: {os.path.join(output_path, f'{name}.csv')}")


Супер-бленд готов! Файл сохранен: /home/jupyter/project/blends/super_blend_leak_at_the_end_v1.csv


In [6]:
# топ-12 предсказаний по private score
files = {
    "logistic_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv",       # 0.69479
    "flaml_automl_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",                   # 0.69348
    "blend_cv_weight_v3": "submissions_v3_leak/submission_blend_cv_weight_search_v3.csv",       # 0.69403
    "blend_greedy_oof_v3": "submissions_v3_leak/submission_blend_greedy_oof_v3.csv",             # 0.69227
    "lgb_eng_v2": "submissions_v2_leak/submission_lgb_top500_engineered.csv",                  # 0.69128
    "blend_prob_equal_v1": "submissions_leak/submission_blend_probability_equal.csv",           # 0.69082
    "blend_logistic_stacker_v2": "submissions_v2_leak/submission_blend_logistic_stacker_rank.csv", # 0.69074
    "lgb_top500_eng_clean_v2": "submissions_v2/submission_lgb_top500_engineered.csv",            # 0.69074
    "blend_equal_rank_topk_v3": "submissions_v3_leak/submission_blend_equal_rank_topk_v3.csv",  # 0.68968
    "logreg_v1": "submissions_leak/submission_logistic_regression.csv",                         # 0.68950
    "log_stacker_v3_clean": "submissions_v3/submission_blend_logistic_stacker_v3.csv",          # 0.68943
    "lgb_dart_v3_clean": "submissions_v3/submission_lgb_dart_v3.csv",                           # 0.68927
}

index_col = "index"
target_col = "score"

print("Загрузка базовых сабмитов...")
dfs = {}
for name, rel_path in files.items():
    full_path = os.path.join(base_path, rel_path)
    if os.path.exists(full_path):
        dfs[name] = pd.read_csv(full_path)
    else:
        # Если вдруг папка с утечкой называется как-то иначе, скрипт возьмет чистую версию как фолбек
        fallback_path = os.path.join(base_path, rel_path.replace("_leak", ""))
        if os.path.exists(fallback_path):
            dfs[name] = pd.read_csv(fallback_path)
            print(f"Предупреждение: файл {rel_path} не найден. Взят фолбек без утечки.")
        else:
            print(f"Ошибка: Файл {rel_path} вообще не найден! Проверь имя.")

# Считаем ранги один раз, чтобы сэкономить ресурсы процессора в цикле
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
model_names = list(ranks.keys())
base_df = next(iter(dfs.values()))[[index_col]].copy()

# Настройки случайного поиска
np.random.seed(1337)  # Элитный сид на удачу
num_submits = 75

print(f"Запуск генерации {num_submits} блендов...")
for i in range(1, num_submits + 1):
    # Случайно выбираем размер подмножества моделей (от 4 до 9 штук для хорошего ансамбля)
    k = np.random.randint(4, 10)
    selected_models = np.random.choice(model_names, size=k, replace=False)
    
    # Генерируем случайные веса (распределение Дирихле дает идеальную сумму весов = 1)
    weights = np.random.dirichlet(np.ones(k))
    
    # Случайная степень для выделения уверенных предсказаний моделей (от 2.5 до 6.5)
    power = round(np.random.uniform(2.5, 6.5), 2)
    
    # Считаем степенной бленд рангов
    final_rank = np.zeros(len(base_df))
    for w, m_name in zip(weights, selected_models):
        final_rank += w * (ranks[m_name] ** power)
        
    # Нормализуем итоговый ранг в финальный скор
    final_score = rankdata(final_rank) / len(final_rank)
    
    sub = base_df.copy()
    sub[target_col] = final_score
    
    # Имя файла содержит метаданные, чтобы ты сразу видел, что круче заходит
    file_name = f"mass_blend_{i:02d}_models_{k}_pow_{power}.csv"
    sub.to_csv(os.path.join(output_path, file_name), index=False)

print(f"Успех! В папке {output_path} лежит {num_submits} готовых файлов.")


Загрузка базовых сабмитов...
Запуск генерации 75 блендов...
Успех! В папке /home/jupyter/project/blends лежит 75 готовых файлов.


In [4]:
# топ-12 предсказаний
files = {
    "logistic_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv",
    "flaml_automl_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "blend_cv_weight_v3": "submissions_v3_leak/submission_blend_cv_weight_search_v3.csv",
    "blend_greedy_oof_v3": "submissions_v3_leak/submission_blend_greedy_oof_v3.csv",
    "lgb_eng_v2": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "blend_prob_equal_v1": "submissions_leak/submission_blend_probability_equal.csv",
    "blend_logistic_stacker_v2": "submissions_v2_leak/submission_blend_logistic_stacker_rank.csv",
    "lgb_top500_eng_clean_v2": "submissions_v2/submission_lgb_top500_engineered.csv",
    "blend_equal_rank_topk_v3": "submissions_v3_leak/submission_blend_equal_rank_topk_v3.csv",
    "logreg_v1": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3_clean": "submissions_v3/submission_blend_logistic_stacker_v3.csv",
    "lgb_dart_v3_clean": "submissions_v3/submission_lgb_dart_v3.csv",
}

index_col = "index"
target_col = "score"

dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items() if os.path.exists(os.path.join(base_path, f))}
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
model_names = list(ranks.keys())
base_df = next(iter(dfs.values()))[[index_col]].copy()

# Настройки направленного поиска
np.random.seed(777)
num_submits = 30

print(f"Генерация {num_submits} сфокусированных блендов...")
for i in range(1, num_submits + 1):
    # Берем только большие ансамбли (8-12 моделей) для стабильности private
    k = np.random.randint(8, len(model_names) + 1)
    selected_models = np.random.choice(model_names, size=k, replace=False)
    
    weights = np.random.dirichlet(np.ones(k))
    
    # Степень строго в золотом диапазоне 2.5 - 3.7
    power = round(np.random.uniform(2.5, 3.7), 2)
    
    final_rank = np.zeros(len(base_df))
    for w, m_name in zip(weights, selected_models):
        final_rank += w * (ranks[m_name] ** power)
        
    final_score = rankdata(final_rank) / len(final_rank)
    
    sub = base_df.copy()
    sub[target_col] = final_score
    
    file_name = f"focused_blend_{i:02d}_models_{k}_pow_{power}.csv"
    sub.to_csv(os.path.join(output_path, file_name), index=False)

print(f"Готово! 30 точечных файлов лежат в {output_path}")

Генерация 30 сфокусированных блендов...
Готово! 30 точечных файлов лежат в /home/jupyter/project/blends


In [5]:
# private > 0.690
files = {
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv",       # 0.69479
    "cv_weight_v3": "submissions_v3_leak/submission_blend_cv_weight_search_v3.csv",         # 0.69403
    "flaml_automl_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",                 # 0.69348
    "greedy_oof_v3": "submissions_v3_leak/submission_blend_greedy_oof_v3.csv",               # 0.69227
    "lgb_eng_v2": "submissions_v2_leak/submission_lgb_top500_engineered.csv",                # 0.69128
    "prob_equal_v1": "submissions_leak/submission_blend_probability_equal.csv",             # 0.69082
    "log_stacker_v2": "submissions_v2_leak/submission_blend_logistic_stacker_rank.csv",       # 0.69074
}

index_col = "index"
target_col = "score"

dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items() if os.path.exists(os.path.join(base_path, f))}
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
model_names = list(ranks.keys())
base_df = next(iter(dfs.values()))[[index_col]].copy()

np.random.seed(999)

# ЧАСТЬ 1: БЛЕНДЫ ВООБЩЕ БЕЗ СТЕПЕНЕЙ (Чистые линейные ранги)
print("Генерация блендов БЕЗ СТЕПЕНЕЙ...")
for i in range(1, 21):
    # Выбираем от 4 до 6 элитных моделей
    k = np.random.randint(4, 7)
    selected_models = np.random.choice(model_names, size=k, replace=False)
    
    # Генерируем веса (сильнейшим моделям - больше веса)
    raw_weights = np.random.uniform(0.1, 1.0, size=k)
    weights = raw_weights / np.sum(raw_weights)
    
    final_rank = np.zeros(len(base_df))
    for w, m_name in zip(weights, selected_models):
        final_rank += w * ranks[m_name]  # Без степеней!
        
    final_score = rankdata(final_rank) / len(final_rank)
    
    sub = base_df.copy()
    sub[target_col] = final_score
    
    file_name = f"elite_linear_blend_{i:02d}_models_{k}.csv"
    sub.to_csv(os.path.join(output_path, file_name), index=False)


# ЧАСТЬ 2: ДИВЕРСИФИЦИРОВАННЫЙ ЭЛИТНЫЙ БЛЕНД СО СТЕПЕНЯМИ (Похож на твой старый топ)
print("Генерация элитных блендов со степенями...")
for i in range(1, 21):
    k = np.random.randint(4, len(model_names) + 1)
    selected_models = np.random.choice(model_names, size=k, replace=False)
    
    weights = np.random.dirichlet(np.ones(k))
    # Держим степень в рамках классических 3.0 - 4.5, как в твоем лучшем решении
    power = round(np.random.uniform(3.0, 4.5), 2)
    
    final_rank = np.zeros(len(base_df))
    for w, m_name in zip(weights, selected_models):
        final_rank += w * (ranks[m_name] ** power)
        
    final_score = rankdata(final_rank) / len(final_rank)
    
    sub = base_df.copy()
    sub[target_col] = final_score
    
    file_name = f"elite_power_blend_{i:02d}_models_{k}_pow_{power}.csv"
    sub.to_csv(os.path.join(output_path, file_name), index=False)

print(f"Готово! В папке {output_path} создано 40 очищенных файлов.")


Генерация блендов БЕЗ СТЕПЕНЕЙ...
Генерация элитных блендов со степенями...
Готово! В папке /home/jupyter/project/blends создано 40 очищенных файлов.


In [6]:
# Загружаем файлы, строго соблюдая наличие утечек там, где они были в старом бленде
# Для новых замен берем их лучшие версии из таблицы
files = {
    # Жесткое ядро старого бленда
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv", # в старом бленде шел БЕЗ утечки
    
    # Старая группа поддержки
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml": "submissions_leak/flaml_all_train_fields_0.66582.csv",
    "old_cat": "submissions_leak/old_catboost.csv",
    
    # Новые мощные кандидаты на замену / усиление
    "lgb_gb": "submissions_leak/submission_lightgbm_gradient_boosting.csv",      # 0.68474!
    "logreg": "submissions_leak/submission_logistic_regression.csv",               # 0.68950!
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv"               # 0.69348!
}

index_col = "index"
target_col = "score"

# Предзагрузка
dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items() if os.path.exists(os.path.join(base_path, f))}
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
base_df = next(iter(dfs.values()))[[index_col]].copy()

# Степень жестко фиксируем около оригинальной (4.5)
power = 4.5

# Вариант 1: Прямая замена old_cat на сильный lgb_gb (Остальное как в старом бленде)
sub1 = base_df.copy()
sub1[target_col] = rankdata(
    0.2525 * (ranks["lgb_eng"]**power + ranks["lgb_dart"]**power) + 
    0.165 * (ranks["tabm"]**power + ranks["lgb_gb"]**power + ranks["flaml"]**power)
) / len(base_df)
sub1.to_csv(os.path.join(output_path, "upgrade_01_cat_to_lgbgb.csv"), index=False)

# Вариант 2: Замена old_cat на взрывную логистическую регрессию
sub2 = base_df.copy()
sub2[target_col] = rankdata(
    0.2525 * (ranks["lgb_eng"]**power + ranks["lgb_dart"]**power) + 
    0.165 * (ranks["tabm"]**power + ranks["logreg"]**power + ranks["flaml"]**power)
) / len(base_df)
sub2.to_csv(os.path.join(output_path, "upgrade_02_cat_to_logreg.csv"), index=False)

# Вариант 3: Замена старого flaml на flaml_v3, а cat на logreg (Максимальный апгрейд поддержки)
sub3 = base_df.copy()
sub3[target_col] = rankdata(
    0.2525 * (ranks["lgb_eng"]**power + ranks["lgb_dart"]**power) + 
    0.165 * (ranks["tabm"]**power + ranks["logreg"]**power + ranks["flaml_v3"]**power)
) / len(base_df)
sub3.to_csv(os.path.join(output_path, "upgrade_03_support_max_boost.csv"), index=False)

# Вариант 4: Добавление logreg шестым элементом с сохранением пропорций ядра
sub4 = base_df.copy()
sub4[target_col] = rankdata(
    0.26 * (ranks["lgb_eng"]**power + ranks["lgb_dart"]**power) + 
    0.12 * (ranks["tabm"]**power + ranks["old_cat"]**power + ranks["flaml"]**power + ranks["logreg"]**power)
) / len(base_df)
sub4.to_csv(os.path.join(output_path, "upgrade_04_add_logreg_6th.csv"), index=False)

# Вариант 5: Добавление lgb_gb шестым элементом
sub5 = base_df.copy()
sub5[target_col] = rankdata(
    0.26 * (ranks["lgb_eng"]**power + ranks["lgb_dart"]**power) + 
    0.12 * (ranks["tabm"]**power + ranks["old_cat"]**power + ranks["flaml"]**power + ranks["lgb_gb"]**power)
) / len(base_df)
sub5.to_csv(os.path.join(output_path, "upgrade_05_add_lgbgb_6th.csv"), index=False)


# Генерируем еще 10 мелких вариаций весов вокруг схемы с заменой Кэтбуста на Логрег/LGB_GB
np.random.seed(42)
for i in range(6, 16):
    # Небольшое качание весов ядра (от 0.22 до 0.28)
    w_core = np.random.uniform(0.22, 0.28)
    # Остаток делим на 3 модели поддержки (tabm, flaml_v3 и либо logreg, либо lgb_gb)
    w_supp = (1.0 - (w_core * 2)) / 3
    
    chosen_cat_repl = "logreg" if i % 2 == 0 else "lgb_gb"
    
    final_rank = (
        w_core * (ranks["lgb_eng"]**power + ranks["lgb_dart"]**power) +
        w_supp * (ranks["tabm"]**power + ranks["flaml_v3"]**power + ranks[chosen_cat_repl]**power)
    )
    
    sub_loop = base_df.copy()
    sub_loop[target_col] = rankdata(final_rank) / len(final_rank)
    sub_loop.to_csv(os.path.join(output_path, f"upgrade_{i:02d}_loop.csv"), index=False)

print(f"Готово! В папке {output_path} создано 15 точечных апгрейдов твоего лучшего бленда.")


Готово! В папке /home/jupyter/project/blends создано 15 точечных апгрейдов твоего лучшего бленда.


In [7]:
# Полный боекомплект лучших проверенных компонентов
files = {
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "logreg": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv"
}

index_col = "index"
target_col = "score"

dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items()}
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
base_df = next(iter(dfs.values()))[[index_col]].copy()

np.random.seed(2026) # Заряжаем сид на прорыв

# --- СТРАТЕГИЯ 1: ПЕРЕВЕРНУТОЕ ЯДРО (12 файлов) ---
# Даем моделям поддержки (особенно логистике и фламлу) доминирующие веса
for i in range(1, 13):
    w_lgb_eng = np.random.uniform(0.14, 0.18)
    w_lgb_dart = np.random.uniform(0.14, 0.18)
    
    # Распределяем оставшийся большой вес между поддержкой
    w_left = 1.0 - (w_lgb_eng + w_lgb_dart)
    raw_supp = np.random.dirichlet(np.ones(3)) # для tabm, flaml_v3, logreg
    w_tabm, w_flaml, w_logreg = raw_supp * w_left
    
    p = round(np.random.uniform(3.5, 5.0), 2)
    
    final_rank = (
        w_lgb_eng * (ranks["lgb_eng"]**p) +
        w_lgb_dart * (ranks["lgb_dart"]**p) +
        w_tabm * (ranks["tabm"]**p) +
        w_flaml * (ranks["flaml_v3"]**p) +
        w_logreg * (ranks["logreg"]**p)
    )
    
    sub = base_df.copy()
    sub[target_col] = rankdata(final_rank) / len(final_rank)
    sub.to_csv(os.path.join(output_path, f"strat1_inverted_core_{i:02d}_p_{p}.csv"), index=False)


# --- СТРАТЕГИЯ 2: ДВОЙНОЕ ЛОГИСТИЧЕСКОЕ УСИЛЕНИЕ (10 файлов) ---
# Смешиваем 6 моделей, где logreg и log_stacker_v3 получают повышенные веса
for i in range(1, 11):
    w_core = np.random.uniform(0.18, 0.22) # lgb_eng и lgb_dart
    w_log_heavy = np.random.uniform(0.18, 0.23) # logreg и log_stacker_v3
    
    w_rest = (1.0 - (w_core * 2) - (w_log_heavy * 2)) / 2 # на tabm и flaml_v3
    
    p = round(np.random.uniform(4.0, 4.8), 2)
    
    final_rank = (
        w_core * (ranks["lgb_eng"]**p + ranks["lgb_dart"]**p) +
        w_log_heavy * (ranks["logreg"]**p + ranks["log_stacker_v3"]**p) +
        w_rest * (ranks["tabm"]**p + ranks["flaml_v3"]**p)
    )
    
    sub = base_df.copy()
    sub[target_col] = rankdata(final_rank) / len(final_rank)
    sub.to_csv(os.path.join(output_path, f"strat2_double_logistic_{i:02d}_p_{p}.csv"), index=False)


# --- СТРАТЕГИЯ 3: АГРЕССИВНЫЙ POWER-СВИНГ (10 файлов) ---
# Фиксируем пропорции лучшего upgrade_10, но бьем по экстремальным степеням
w_core_fixed = 0.225
w_supp_fixed = 0.183
extreme_powers = [1.5, 2.0, 2.5, 3.0, 5.2, 5.5, 6.0, 6.5, 7.0, 8.0]

for idx, p in enumerate(extreme_powers, 1):
    final_rank = (
        w_core_fixed * (ranks["lgb_eng"]**p + ranks["lgb_dart"]**p) +
        w_supp_fixed * (ranks["tabm"]**p + ranks["flaml_v3"]**p + ranks["logreg"]**p)
    )
    sub = base_df.copy()
    sub[target_col] = rankdata(final_rank) / len(final_rank)
    sub.to_csv(os.path.join(output_path, f"strat3_power_swing_{idx:02d}_p_{p}.csv"), index=False)

print(f"Пайплайн завершен! Ровно 32 агрессивных снаряда готовы к отправке в {output_path}")

Пайплайн завершен! Ровно 32 агрессивных снаряда готовы к отправке в /home/jupyter/project/blends


In [8]:
files = {
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "logreg": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv"
}

index_col = "index"
target_col = "score"

dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items()}
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
base_df = next(iter(dfs.values()))[[index_col]].copy()

def save_sub(final_rank, filename):
    final_score = rankdata(final_rank) / len(final_rank)
    sub = base_df.copy()
    sub[target_col] = final_score
    sub.to_csv(os.path.join(output_path, filename), index=False)

# ---- ШОТ 1: Уклон в жесткое бустинг-ядро (повышаем резкость) ----
p1 = 4.15
rank_1 = (
    0.23 * (ranks["lgb_eng"]**p1 + ranks["lgb_dart"]**p1) +
    0.18 * (ranks["logreg"]**p1 + ranks["log_stacker_v3"]**p1) +
    0.09 * (ranks["tabm"]**p1 + ranks["flaml_v3"]**p1)
)
save_sub(rank_1, "final_shot_01_heavy_core_p4.15.csv")

# ---- ШОТ 2: Уклон в двойную логистику (калибровка на максимум) ----
p2 = 4.10
rank_2 = (
    0.18 * (ranks["lgb_eng"]**p2 + ranks["lgb_dart"]**p2) +
    0.23 * (ranks["logreg"]**p2 + ranks["log_stacker_v3"]**p2) +
    0.09 * (ranks["tabm"]**p2 + ranks["flaml_v3"]**p2)
)
save_sub(rank_2, "final_shot_02_heavy_log_p4.10.csv")

# ---- ШОТ 3: Золотая середина с пониженной степенью ----
p3 = 3.95
rank_3 = (
    0.20 * (ranks["lgb_eng"]**p3 + ranks["lgb_dart"]**p3) +
    0.20 * (ranks["logreg"]**p3 + ranks["log_stacker_v3"]**p3) +
    0.10 * (ranks["tabm"]**p3 + ranks["flaml_v3"]**p3)
)
save_sub(rank_3, "final_shot_03_balanced_p3.95.csv")

# ---- ШОТ 4: Усиление за счет FLAML_v3 (взято из паттернов Strategy 1) ----
p4 = 4.05
rank_4 = (
    0.19 * (ranks["lgb_eng"]**p4 + ranks["lgb_dart"]**p4) +
    0.19 * (ranks["logreg"]**p4 + ranks["log_stacker_v3"]**p4) +
    0.07 * (ranks["tabm"]**p4) + 0.17 * (ranks["flaml_v3"]**p4)
)
save_sub(rank_4, "final_shot_04_flaml_boost_p4.05.csv")

# ---- ШОТ 5: Мягкое сглаживание (минимальная степень для борьбы с переобучением) ----
p5 = 3.85
rank_5 = (
    0.21 * (ranks["lgb_eng"]**p5 + ranks["lgb_dart"]**p5) +
    0.21 * (ranks["logreg"]**p5 + ranks["log_stacker_v3"]**p5) +
    0.08 * (ranks["tabm"]**p5 + ranks["flaml_v3"]**p5)
)
save_sub(rank_5, "final_shot_05_smooth_p3.85.csv")

print(f"Все 5 элитных финальных сабмитов лежат в {output_path}!")


Все 5 элитных финальных сабмитов лежат в /home/jupyter/project/blends!


### Следующее

In [ ]:
# ПОЛНЫЙ БОЕКОМПЛЕКТ ИЗ 12 МОДЕЛЕЙ (Все на месте, никто не забыт)
files = {
    "lgb_eng": "submissions_v2_leak/submission_lgb_top500_engineered.csv",
    "lgb_dart": "submissions_v2/submission_lgb_peak_dart.csv",
    "tabm": "submissions_leak/tabm_top500_magic_meta_0.65617.csv",
    "flaml": "submissions_leak/flaml_all_train_fields_0.66582.csv",
    "flaml_v3": "submissions_v3_leak/submission_flaml_automl_v3.csv",
    "logreg": "submissions_leak/submission_logistic_regression.csv",
    "log_stacker_v3": "submissions_v3_leak/submission_blend_logistic_stacker_v3.csv",
    "greedy_oof_v3": "submissions_v3_leak/submission_blend_greedy_oof_v3.csv",
    "cv_weight_v3": "submissions_v3_leak/submission_blend_cv_weight_search_v3.csv",
    "cat_linear_stack": "submissions_v2_leak/submission_catboost_linear_stack_compact.csv",
    "cat_depth6_v3": "submissions_v3_leak/submission_catboost_depth6_v3.csv",
    "xgb_pca_stack": "submissions_v2_leak/submission_xgb_pca_svd_stack.csv"
}

index_col = "index"
target_col = "score"

print("Загрузка полного пула из 12 моделей...")
dfs = {name: pd.read_csv(os.path.join(base_path, f)) for name, f in files.items() if os.path.exists(os.path.join(base_path, f))}
ranks = {name: rankdata(df[target_col]) / len(df) for name, df in dfs.items()}
model_names = list(ranks.keys())
base_df = next(iter(dfs.values()))[[index_col]].copy()

# Настройки генерации
np.random.seed(555) # Новый сид на победу
num_submits = 60

print(f"Запуск генерации {num_submits} мега-блендов...")
for i in range(1, num_submits + 1):
    # Берем от 7 до 10 моделей, чтобы ансамбли были максимально плотными и разнообразными
    k = np.random.randint(7, 11)
    selected_models = np.random.choice(model_names, size=k, replace=False)
    
    # Генерируем случайные веса через Дирихле
    weights = np.random.dirichlet(np.ones(k))
    
    # Степень в проверенном золотом коридоре 3.7 - 4.5
    power = round(np.random.uniform(3.7, 4.5), 2)
    
    # Считаем степенной бленд рангов
    final_rank = np.zeros(len(base_df))
    for w, m_name in zip(weights, selected_models):
        final_rank += w * (ranks[m_name] ** power)
        
    final_score = rankdata(final_rank) / len(final_rank)
    
    sub = base_df.copy()
    sub[target_col] = final_score
    
    # Тэги в названии файла для удобного анализа результатов
    has_tabm = "tabm" if "tabm" in selected_models else "notabm"
    has_cat = "cat" if any("cat" in m for m in selected_models) else "nocat"
    
    file_name = f"mega_{i:02d}_m_{k}_pow_{power}_{has_tabm}_{has_cat}.csv"
    sub.to_csv(os.path.join(output_path, file_name), index=False)

print(f"Готово! 60 сбалансированных файлов сгенерированы в папку {output_path}")
